In [ ]:
!pip -q install yt-dlp ffmpeg-python
!apt-get -qq install ffmpeg
import os
import yt_dlp
import ffmpeg
from pathlib import Path

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.2 MB/s eta 0:00:00


In [ ]:
OUTPUT_DIR = Path("downloads")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
def download_audio(youtube_url, output_dir="downloads"):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    output_template = str(output_dir / "%(id)s.%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_template,
        "quiet": False,
        "noplaylist": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)

    downloaded_file = None

    for file in output_dir.iterdir():
        if file.stem == info["id"]:
            downloaded_file = file
            break

    return downloaded_file, info

In [ ]:
def convert_to_wav(input_file, output_dir="downloads"):
    output_dir = Path(output_dir)

    output_file = output_dir / f"{input_file.stem}.wav"

    (
        ffmpeg
        .input(str(input_file))
        .output(
            str(output_file),
            ac=1,          # mono
            ar=16000,      # 16 kHz
            format="wav"
        )
        .overwrite_output()
        .run(quiet=True)
    )

    return output_file

In [ ]:
youtube_url = "https://youtu.be/keeqnciDVOo?si=Ca9bwY9pOcBW48fq"

audio_file, metadata = download_audio(youtube_url)

wav_file = convert_to_wav(audio_file)

print("Title :", metadata["title"])
print("Duration :", metadata["duration"], "seconds")
print("Saved :", wav_file)

[youtube] Extracting URL: https://youtu.be/keeqnciDVOo?si=Ca9bwY9pOcBW48fq
[youtube] keeqnciDVOo: Downloading webpage


[youtube] keeqnciDVOo: Downloading android vr player API JSON
[info] keeqnciDVOo: Downloading 1 format(s): 251
[download] Destination: downloads/keeqnciDVOo.webm
[download] 100% of    2.50MiB in 00:00:00 at 4.15MiB/s   
Title : Computer Networking in 100 Seconds
Duration : 137 seconds
Saved : downloads/keeqnciDVOo.wav


In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA Available: True
GPU: Tesla T4


In [ ]:
#normalizing the audio
from pathlib import Path
import ffmpeg

def normalize_audio(input_audio):
    """
    Normalize audio loudness using FFmpeg's loudnorm filter.

    Args:
        input_audio (str or Path): Input WAV file.

    Returns:
        Path: Path to normalized WAV file.
    """

    input_audio = Path(input_audio)

    output_audio = input_audio.with_name(
        input_audio.stem + "_normalized.wav"
    )

    (
        ffmpeg
        .input(str(input_audio))
        .output(
            str(output_audio),
            af="loudnorm=I=-16:LRA=11:TP=-1.5"
        )
        .overwrite_output()
        .run(quiet=True)
    )

    return output_audio

In [ ]:
normalized_audio = normalize_audio(wav_file)

print(normalized_audio)

downloads/keeqnciDVOo_normalized.wav


In [ ]:
!pip install silero-vad torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.5 MB/s eta 0:00:00


In [ ]:
import torch
import torchaudio

from silero_vad import (
    load_silero_vad,
    read_audio,
    get_speech_timestamps,
    collect_chunks
)

In [ ]:
#silence removal
def remove_silence_vad(input_audio):
    """
    Removes silence using Silero VAD.

    Returns
    -------
    cleaned_audio_path
    speech_timestamps
    """

    input_audio = Path(input_audio)

    output_audio = input_audio.with_name(
        input_audio.stem + "_speech.wav"
    )

    model = load_silero_vad()

    wav = read_audio(str(input_audio), sampling_rate=16000)

    speech_timestamps = get_speech_timestamps(
        wav,
        model,
        sampling_rate=16000,
        threshold=0.5,
        min_speech_duration_ms=250,
        min_silence_duration_ms=500,
        speech_pad_ms=200,
    )

    speech = collect_chunks(
        speech_timestamps,
        wav
    )

    torchaudio.save(
        str(output_audio),
        speech.unsqueeze(0),
        16000
    )

    return output_audio, speech_timestamps

In [ ]:
speech_audio, timestamps = remove_silence_vad(normalized_audio)

print(speech_audio)

print(timestamps[:5])

downloads/keeqnciDVOo_normalized_speech.wav
[{'start': 384, 'end': 2195865}]


In [ ]:
def timestamps_to_seconds(timestamps, sample_rate=16000):

    return [
        {
            "start": round(t["start"] / sample_rate, 2),
            "end": round(t["end"] / sample_rate, 2)
        }
        for t in timestamps
    ]

In [ ]:
segments = timestamps_to_seconds(timestamps)

segments[:10]

[{'start': 0.02, 'end': 137.24}]

In [ ]:
#everyhting at once
from pathlib import Path

def preprocess_audio(youtube_url):
    """
    Complete audio preprocessing pipeline.

    Steps
    -----
    1. Download best available audio
    2. Convert to 16kHz Mono WAV
    3. Normalize loudness
    4. Remove silence using Silero VAD

    Parameters
    ----------
    youtube_url : str

    Returns
    -------
    dict
        {
            "title": ...,
            "duration": ...,
            "clean_audio": Path(...),
            "timestamps": [...],
            "metadata": {...}
        }
    """

    print("=" * 60)
    print("Downloading audio...")
    audio_file, metadata = download_audio(youtube_url)

    print("Converting to WAV...")
    wav_file = convert_to_wav(audio_file)

    print("Normalizing audio...")
    normalized_audio = normalize_audio(wav_file)

    print("Removing silence...")
    clean_audio, timestamps = remove_silence_vad(normalized_audio)

    print("=" * 60)
    print("Audio preprocessing completed!")
    print("=" * 60)

    return {
        "title": metadata["title"],
        "duration": metadata["duration"],
        "clean_audio": clean_audio,
        "timestamps": timestamps_to_seconds(timestamps),
        "metadata": metadata
    }

In [ ]:
youtube_url = "https://youtu.be/keeqnciDVOo?si=Ca9bwY9pOcBW48fq"

result = preprocess_audio(youtube_url)

print(result["title"])

print(result["duration"])

print(result["clean_audio"])

print(result["timestamps"][:5])

[youtube] Extracting URL: https://youtu.be/keeqnciDVOo?si=Ca9bwY9pOcBW48fq
[youtube] keeqnciDVOo: Downloading webpage


[youtube] keeqnciDVOo: Downloading android vr player API JSON
[info] keeqnciDVOo: Downloading 1 format(s): 251
[download] Destination: downloads/keeqnciDVOo.webm
[download] 100% of    2.50MiB in 00:00:00 at 15.11MiB/s  
Converting to WAV...
Normalizing audio...
Removing silence...
Audio preprocessing completed!
Computer Networking in 100 Seconds
137
downloads/keeqnciDVOo_normalized_speech.wav
[{'start': 0.02, 'end': 137.24}]


In [ ]:
!pip install -q faster-whisper
from faster_whisper import WhisperModel
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.2 MB/s eta 0:00:00


In [ ]:
device = "cuda"

model = WhisperModel(
    "large-v3",
    device=device,
    compute_type="float16"
)

In [ ]:
def transcribe_audio(
    audio_path,
    output_dir="downloads/transcript",
    language=None
):
    """
    Transcribe audio using Faster-Whisper.

    Features
    --------
    ✓ Prints transcript in Colab
    ✓ Saves transcript as TXT
    ✓ Saves timestamps internally

    Returns
    -------
    transcript_text
    segments_data
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    segments, info = model.transcribe(
        str(audio_path),
        beam_size=5,
        vad_filter=False,      # already used Silero
        language=language
    )

    transcript = []
    segments_data = []

    print("=" * 70)
    print("TRANSCRIPT")
    print("=" * 70)

    for segment in segments:

        text = segment.text.strip()

        transcript.append(text)

        segments_data.append({
            "start": round(segment.start, 2),
            "end": round(segment.end, 2),
            "text": text
        })

        print(text)

    transcript_text = "\n".join(transcript)

    txt_file = output_dir / "transcript.txt"

    with open(txt_file, "w", encoding="utf-8") as f:
        f.write(transcript_text)

    print("\n")
    print("=" * 70)
    print("Language :", info.language)
    print("Duration :", round(info.duration,2), "seconds")
    print("Saved to :", txt_file)

    return transcript_text, segments_data

In [ ]:
result = preprocess_audio(youtube_url)

transcript, segments = transcribe_audio(
    result["clean_audio"],
    language="en"
)

[youtube] Extracting URL: https://youtu.be/keeqnciDVOo?si=Ca9bwY9pOcBW48fq
[youtube] keeqnciDVOo: Downloading webpage


[youtube] keeqnciDVOo: Downloading android vr player API JSON
[info] keeqnciDVOo: Downloading 1 format(s): 251
[download] Destination: downloads/keeqnciDVOo.webm
[download] 100% of    2.50MiB in 00:00:00 at 17.01MiB/s  
Converting to WAV...
Normalizing audio...
Removing silence...
Audio preprocessing completed!
TRANSCRIPT
Networking. It's the way computers exchange information around the world. And just like
the burrito, its architecture is abstracted into seven layers based on the open systems
interconnection model. At the bottom, we have physical hardware, like fiber optic cables that
literally carry light from point A to point B. Somehow, this light travels all the way to layer
seven, where it can be transmitted directly into your consciousness in the form of pixels on a
screen or vibrations from a speaker. That's exactly what you're doing right now as an end
user accessing this video over the hypertext transfer protocol. In addition to HTTP,
there are many other protocols at layer 

In [ ]:
#done
